In [ ]:
%run canvas/OEA_py_canvas

In [ ]:
import aiohttp
import asyncio
import json
from datetime import date
from notebookutils import mssparkutils

In [ ]:

class CanvasRawIngestion:
    """
    Canvas LMS → OEA Stage-1 RAW ingestion
    Fetches:
    accounts, users, courses, sections, enrollments,
    assignments, assignment_submissions,
    quizzes, quiz_questions, quiz_submissions
    """

    def __init__(self, canvas_host, canvas_token):
        self.host = canvas_host
        self.token = canvas_token
        self.rundate = date.today().isoformat()
        self.base_path = f"stage1/Transactional/canvas_raw/v0.1"
        self.session = None

    # ---------------- HTTP ----------------

    async def _open(self):
        self.session = aiohttp.ClientSession(
            headers={
                "Authorization": f"Bearer {self.token}",
                "Content-Type": "application/json"
            }
        )

    async def _close(self):
        await self.session.close()

    async def _get(self, path):
        url = f"https://{self.host}/{path.lstrip('/')}"
        async with self.session.get(url) as r:
            r.raise_for_status()
            return await r.json()

    # ---------------- Storage ----------------

    def _path(self, entity):
        return oea.to_url(
            self.base_path,
            entity,
            "delta_batch_data",
            f"rundate={self.rundate}"
        )

    def _write(self, entity, filename, data):
        path = self._path(entity)
        mssparkutils.fs.mkdirs(path)

        full_path = f"{path}/{filename}"
        json_data = json.dumps(data, indent=2)

        mssparkutils.fs.put(full_path, json_data, overwrite=True)

    # ---------------- ROOT ----------------

    async def ingest_accounts(self):
        data = await self._get("/api/v1/accounts")
        self._write("Accounts", "accounts.json", data)
        return data

    async def ingest_users(self, account_id):
        data = await self._get(f"/api/v1/accounts/{account_id}/users")
        self._write("Users", f"users_{account_id}.json", data)
        return data

    async def ingest_courses(self, account_id):
        data = await self._get(f"/api/v1/accounts/{account_id}/courses")
        self._write("Courses", f"courses_{account_id}.json", data)
        return data

    # ---------------- COURSE LEVEL ----------------

    async def ingest_sections(self, course_id):
        data = await self._get(f"/api/v1/courses/{course_id}/sections")
        self._write("Sections", f"sections_{course_id}.json", data)
        return data

    async def ingest_quizzes(self, course_id):
        data = await self._get(f"/api/v1/courses/{course_id}/quizzes")
        self._write("Quizzes", f"quizzes_{course_id}.json", data)
        return data

    # ---------------- SECTION LEVEL ----------------

    async def ingest_enrollments(self, section_id):
        data = await self._get(f"/api/v1/sections/{section_id}/enrollments")
        self._write("Enrollments", f"enrollments_{section_id}.json", data)
        return data

    # ---------------- QUIZ LEVEL ----------------

    async def ingest_quiz_questions(self, course_id, quiz_id):
        data = await self._get(
            f"/api/v1/courses/{course_id}/quizzes/{quiz_id}/questions"
        )
        self._write(
            "Quiz_Questions",
            f"questions_{course_id}_{quiz_id}.json",
            data
        )

    async def ingest_quiz_submissions(self, course_id, quiz_id):
        data = await self._get(
            f"/api/v1/courses/{course_id}/quizzes/{quiz_id}/submissions"
        )
        self._write(
            "Quiz_Submissions",
            f"quiz_submissions_{course_id}_{quiz_id}.json",
            data
        )

    # ---------------- MASTER PIPELINE ----------------

    async def ingest_all(self):
        await self._open()
        try:
            accounts = await self.ingest_accounts()

            for acc in accounts:
                acc_id = acc["id"]

                users = await self.ingest_users(acc_id)
                courses = await self.ingest_courses(acc_id)

                for course in courses:
                    course_id = course["id"]

                    sections = await self.ingest_sections(course_id)
                    quizzes = await self.ingest_quizzes(course_id)

                    for s in sections:
                        await self.ingest_enrollments(s["id"])

                    for q in quizzes:
                        await self.ingest_quiz_questions(course_id, q["id"])
                        await self.ingest_quiz_submissions(course_id, q["id"])

            return {"status": "canvas_raw_complete"}

        finally:
            await self._close()

CRIngest = CanvasRawIngestion("canvas_host", "canvas_token")

In [ ]:
# ############## Incremental load handling ####################
# import aiohttp
# import json
# from datetime import datetime, timezone, timedelta
# from datetime import date

# # Synapse utility
# from notebookutils import mssparkutils

# import oea


# class CanvasRawIngestion:
#     """
#     Canvas LMS → OEA Stage-1 RAW ingestion (INCREMENTAL)
#     Fetches:
#     accounts, users, courses, sections, enrollments,
#     quizzes, quiz_questions,
#     quiz_submissions (INCREMENTAL)
#     """

#     WATERMARK_PATH = "stage1/Transactional/canvas_raw/v0.1/watermarks"

#     def __init__(self, canvas_host, canvas_token):
#         self.host = canvas_host
#         self.token = canvas_token
#         self.rundate = date.today().isoformat()
#         self.base_path = f"stage1/Transactional/canvas_raw/v0.1"
#         self.session = None

#     # ---------------- TIME ----------------

#     def _utc_now(self):
#         return datetime.now(timezone.utc).isoformat()

#     # ---------------- HTTP ----------------

#     async def _open(self):
#         self.session = aiohttp.ClientSession(
#             headers={
#                 "Authorization": f"Bearer {self.token}",
#                 "Content-Type": "application/json"
#             }
#         )

#     async def _close(self):
#         if self.session:
#             await self.session.close()

#     async def _get(self, path, params=None):
#         url = f"https://{self.host}/{path.lstrip('/')}"
#         async with self.session.get(url, params=params) as r:
#             r.raise_for_status()
#             return await r.json()

#     # ---------------- STORAGE ----------------

#     def _path(self, entity):
#         return oea.to_url(
#             self.base_path,
#             entity,
#             "delta_batch_data",
#             f"rundate={self.rundate}"
#         )

#     def _write(self, entity, filename, data):
#         path = self._path(entity)
#         mssparkutils.fs.mkdirs(path)

#         full_path = f"{path}/{filename}"
#         json_data = json.dumps(data, indent=2)

#         mssparkutils.fs.put(full_path, json_data, overwrite=True)

#     # ---------------- WATERMARK ----------------

#     def _watermark_file(self, entity):
#         return f"{self.WATERMARK_PATH}/{entity}.json"

#     def _read_watermark(self, entity):
#         try:
#             path = self._watermark_file(entity)
#             if mssparkutils.fs.exists(path):
#                 raw = mssparkutils.fs.head(path, 1024)
#                 return json.loads(raw)["last_run"]
#         except Exception:
#             pass

#         # First run fallback = last 7 days
#         return (datetime.now(timezone.utc) - timedelta(days=7)).isoformat()

#     def _write_watermark(self, entity, timestamp):
#         mssparkutils.fs.mkdirs(self.WATERMARK_PATH)

#         payload = {
#             "entity": entity,
#             "last_run": timestamp
#         }

#         mssparkutils.fs.put(
#             self._watermark_file(entity),
#             json.dumps(payload, indent=2),
#             overwrite=True
#         )

#     # ---------------- PAGINATION ----------------

#     async def _get_all_pages(self, path, params=None):
#         """
#         Canvas paginates most endpoints.
#         This safely walks through all pages.
#         """
#         results = []
#         page = 1
#         params = params or {}

#         while True:
#             params["page"] = page
#             params["per_page"] = 100

#             batch = await self._get(path, params=params)
#             if not batch:
#                 break

#             results.extend(batch)
#             page += 1

#         return results

#     # ---------------- ROOT ----------------

#     async def ingest_accounts(self):
#         data = await self._get_all_pages("/api/v1/accounts")
#         self._write("Accounts", "accounts.json", data)
#         return data

#     async def ingest_users(self, account_id):
#         data = await self._get_all_pages(
#             f"/api/v1/accounts/{account_id}/users"
#         )
#         self._write("Users", f"users_{account_id}.json", data)
#         return data

#     async def ingest_courses(self, account_id):
#         data = await self._get_all_pages(
#             f"/api/v1/accounts/{account_id}/courses"
#         )
#         self._write("Courses", f"courses_{account_id}.json", data)
#         return data

#     # ---------------- COURSE LEVEL ----------------

#     async def ingest_sections(self, course_id):
#         data = await self._get_all_pages(
#             f"/api/v1/courses/{course_id}/sections"
#         )
#         self._write("Sections", f"sections_{course_id}.json", data)
#         return data

#     async def ingest_quizzes(self, course_id):
#         data = await self._get_all_pages(
#             f"/api/v1/courses/{course_id}/quizzes"
#         )
#         self._write("Quizzes", f"quizzes_{course_id}.json", data)
#         return data

#     # ---------------- SECTION LEVEL ----------------

#     async def ingest_enrollments(self, section_id):
#         data = await self._get_all_pages(
#             f"/api/v1/sections/{section_id}/enrollments"
#         )
#         self._write("Enrollments", f"enrollments_{section_id}.json", data)
#         return data

#     # ---------------- QUIZ LEVEL ----------------

#     async def ingest_quiz_questions(self, course_id, quiz_id):
#         data = await self._get_all_pages(
#             f"/api/v1/courses/{course_id}/quizzes/{quiz_id}/questions"
#         )
#         self._write(
#             "Quiz_Questions",
#             f"questions_{course_id}_{quiz_id}.json",
#             data
#         )
#         return data

#     async def ingest_quiz_submissions(self, course_id, quiz_id):
#         """
#         INCREMENTAL:
#         Only fetch submissions updated since last successful run
#         """
#         last_run = self._read_watermark("Quiz_Submissions")

#         params = {
#             "updated_since": last_run
#         }

#         data = await self._get_all_pages(
#             f"/api/v1/courses/{course_id}/quizzes/{quiz_id}/submissions",
#             params=params
#         )

#         if data:
#             self._write(
#                 "Quiz_Submissions",
#                 f"quiz_submissions_{course_id}_{quiz_id}.json",
#                 data
#             )

#         # Update watermark only after success
#         self._write_watermark("Quiz_Submissions", self._utc_now())

#         return data

#     # ---------------- MASTER PIPELINE ----------------

#     async def ingest_all(self):
#         await self._open()

#         try:
#             accounts = await self.ingest_accounts()

#             for acc in accounts:
#                 acc_id = acc["id"]

#                 await self.ingest_users(acc_id)
#                 courses = await self.ingest_courses(acc_id)

#                 for course in courses:
#                     course_id = course["id"]

#                     sections = await self.ingest_sections(course_id)
#                     quizzes = await self.ingest_quizzes(course_id)

#                     for s in sections:
#                         await self.ingest_enrollments(s["id"])

#                     for q in quizzes:
#                         await self.ingest_quiz_questions(course_id, q["id"])
#                         await self.ingest_quiz_submissions(course_id, q["id"])

#             return {"status": "canvas_raw_incremental_complete"}

#         finally:
#             await self._close()


In [ ]:
# ############### Github Testing ###############
# import aiohttp
# import json
# from datetime import date
# from notebookutils import mssparkutils


# class CanvasRawIngestion:
#     """
#     Canvas LMS → OEA Stage-1 RAW ingestion (TEST MODE)
#     Source: GitHub raw JSON test files
#     """

#     BASE_GITHUB_URL = (
#         "https://raw.githubusercontent.com/"
#         "vaibhav01062003/canvas-lms-etl-pipeline/"
#         "refs/heads/main/test_data"
#     )

#     FILE_MAP = {
#         "Accounts": "Account.json",
#         "Users": "User.json",
#         "Courses": "Course.json",
#         "Sections": "Section.json",
#         "Enrollments": "Enrollement.json",
#         "Quizzes": "Quiz.json",
#         "Quiz_Questions": "Quiz%20Question.json",
#         "Quiz_Submissions": "Quiz%20Submission.json",
#         "Outcomes": "outcomes.json",
#     }

#     def __init__(self):
#         self.rundate = date.today().isoformat()
#         self.base_path = f"stage1/Transactional/canvas_raw/v0.1"
#         self.session = None

#     # ---------------- HTTP ----------------

#     async def _open(self):
#         self.session = aiohttp.ClientSession()

#     async def _close(self):
#         await self.session.close()

#     async def _get_file(self, filename):
#         url = f"{self.BASE_GITHUB_URL}/{filename}"
#         async with self.session.get(url) as r:
#             r.raise_for_status()
#             return await r.json(content_type=None)

#     # ---------------- Storage ----------------

#     def _path(self, entity):
#         full_path = (
#             f"{self.base_path}/"
#             f"{entity}/"
#             f"delta_batch_data/"
#             f"rundate={self.rundate}"
#         )
#         return oea.to_url(full_path)


#     def _write(self, entity, filename, data):
#         path = self._path(entity)
#         mssparkutils.fs.mkdirs(path)

#         full_path = f"{path}/{filename}"
#         json_data = json.dumps(data, indent=2)

#         mssparkutils.fs.put(full_path, json_data, overwrite=True)


#     # ---------------- INGEST METHODS ----------------

#     async def ingest_accounts(self):
#         data = await self._get_file(self.FILE_MAP["Accounts"])
#         self._write("Accounts", "accounts.json", data)
#         return data

#     async def ingest_users(self):
#         data = await self._get_file(self.FILE_MAP["Users"])
#         self._write("Users", "users.json", data)
#         return data

#     async def ingest_courses(self):
#         data = await self._get_file(self.FILE_MAP["Courses"])
#         self._write("Courses", "courses.json", data)
#         return data

#     async def ingest_sections(self):
#         data = await self._get_file(self.FILE_MAP["Sections"])
#         self._write("Sections", "sections.json", data)
#         return data

#     async def ingest_enrollments(self):
#         data = await self._get_file(self.FILE_MAP["Enrollments"])
#         self._write("Enrollments", "enrollments.json", data)
#         return data

#     async def ingest_quizzes(self):
#         data = await self._get_file(self.FILE_MAP["Quizzes"])
#         self._write("Quizzes", "quizzes.json", data)
#         return data

#     async def ingest_quiz_questions(self):
#         data = await self._get_file(self.FILE_MAP["Quiz_Questions"])
#         self._write("Quiz_Questions", "quiz_questions.json", data)
#         return data

#     async def ingest_quiz_submissions(self):
#         data = await self._get_file(self.FILE_MAP["Quiz_Submissions"])
#         self._write("Quiz_Submissions", "quiz_submissions.json", data)
#         return data

#     async def ingest_outcomes(self):
#         data = await self._get_file(self.FILE_MAP["Outcomes"])
#         self._write("Outcomes", "outcomes.json", data)
#         return data

#     # ---------------- MASTER PIPELINE ----------------

#     async def ingest_all(self):
#         await self._open()
#         try:
#             await self.ingest_accounts()
#             await self.ingest_users()
#             await self.ingest_courses()
#             await self.ingest_sections()
#             await self.ingest_enrollments()
#             await self.ingest_quizzes()
#             await self.ingest_quiz_questions()
#             await self.ingest_quiz_submissions()
#             await self.ingest_outcomes()

#             return {"status": "canvas_raw_testdata_complete"}

#         finally:
#             await self._close()

# CRIngest = CanvasRawIngestion()

In [ ]:
# await CRIngest.ingest_all()